# IDRiD Diabetic Retinopathy — Stage 2: Local / Centralized EfficientNet-B4 Training

## Stage 2 — Local/Centralized Training

Now that Stage 1 preprocessing + EDA is complete, the next step is **local training only**.

We will:

```text
Processed IDRiD
      ↓
EfficientNet-B4
      ↓
5-class classification
      ↓
Train split
      ↓
Validation split
      ↓
Select best model
      ↓
Official 103-image test
```

**Do not use FedAvg or DP yet.**

Run these cells one by one.

### Cell 1 — Imports and configuration

In [ ]:
# Cell 1: Stage 2 configuration

import os
import json
import copy
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42

NUM_CLASSES = 5
IMAGE_SIZE = 512
BATCH_SIZE = 4

NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

PATIENCE = 8

NUM_WORKERS = 2

OUTPUT_DIR = Path("./processed_idrid")
METADATA_DIR = OUTPUT_DIR / "metadata"
EDA_DIR = OUTPUT_DIR / "eda"
MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### Cell 2 — Load Stage 1 metadata

In [ ]:
# Cell 2: Load processed metadata

train_df = pd.read_csv(
    METADATA_DIR / "train.csv"
)

val_df = pd.read_csv(
    METADATA_DIR / "val.csv"
)

test_df = pd.read_csv(
    METADATA_DIR / "test.csv"
)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

print("\nTraining distribution:")
print(
    train_df["Retinopathy grade"]
    .value_counts()
    .sort_index()
)

print("\nValidation distribution:")
print(
    val_df["Retinopathy grade"]
    .value_counts()
    .sort_index()
)

print("\nTest distribution:")
print(
    test_df["Retinopathy grade"]
    .value_counts()
    .sort_index()
)

### Cell 3 — Recreate transforms

In [ ]:
# Cell 3: Recreate Stage 1 transforms

from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print("Transforms ready")

### Cell 4 — IDRiD Dataset class

In [ ]:
# Cell 4: Reusable IDRiD dataset

from torch.utils.data import Dataset

class IDRiDDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        label = int(
            row["Retinopathy grade"]
        )

        if self.transform:
            image = self.transform(image)

        return image, label

### Cell 5 — Create datasets

In [ ]:
# Cell 5: Create datasets

train_dataset = IDRiDDataset(
    train_df,
    transform=train_transform
)

val_dataset = IDRiDDataset(
    val_df,
    transform=eval_transform
)

test_dataset = IDRiDDataset(
    test_df,
    transform=eval_transform
)

print("Train dataset:", len(train_dataset))
print("Val dataset  :", len(val_dataset))
print("Test dataset :", len(test_dataset))

### Cell 6 — DataLoaders

In [ ]:
# Cell 6: Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

images, labels = next(iter(train_loader))

print("Image batch:", images.shape)
print("Label batch:", labels.shape)

assert images.shape[1:] == (
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

print("DataLoaders working")

### Cell 7 — Calculate class weights

Use **only the training split**.

In [ ]:
# Cell 7: Calculate class weights

from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2, 3, 4])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["Retinopathy grade"].values
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

print("Class weights:")

for cls, weight in zip(
    classes,
    class_weights
):
    print(
        f"Grade {cls}: {weight:.4f}"
    )

### Cell 8 — Build EfficientNet-B4

In [ ]:
# Cell 8: EfficientNet-B4 model

weights = models.EfficientNet_B4_Weights.DEFAULT
model = models.efficientnet_b4(weights=weights)

# Replace final classification layer
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(DEVICE)
print(model.classifier)

### Cell 9 — Loss, optimizer, scheduler

In [ ]:
# Cell 9: Loss + optimizer + scheduler

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(alpha=class_weights_tensor, gamma=2.0)

# Differential Learning Rates
backbone_params = []
head_params = []

for name, param in model.named_parameters():
    if 'classifier' in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LEARNING_RATE * 0.1},
    {'params': head_params, 'lr': LEARNING_RATE}
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    T_mult=2
)

print("Loss:", criterion.__class__.__name__)
print("Optimizer:", optimizer.__class__.__name__)
print("Scheduler:", scheduler.__class__.__name__)

### Cell 10 — Training function

In [ ]:
# Cell 10: Training function with Mixup

def mixup_data(x, y, alpha=0.2, device='cuda'):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.2, device=device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        
        loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.detach().cpu().numpy())
        all_predictions.extend(predictions.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_accuracy = accuracy_score(all_labels, all_predictions)
    return epoch_loss, epoch_accuracy

### Cell 11 — Validation function

In [ ]:
# Cell 11: Validation function with Test-Time Augmentation (TTA)

def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Standard prediction
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            
            probabilities = torch.softmax(outputs, dim=1)

            # TTA: Horizontal Flip
            images_hflip = torch.flip(images, dims=[3])
            outputs_hflip = model(images_hflip)
            probabilities_hflip = torch.softmax(outputs_hflip, dim=1)

            # TTA: Vertical Flip
            images_vflip = torch.flip(images, dims=[2])
            outputs_vflip = model(images_vflip)
            probabilities_vflip = torch.softmax(outputs_vflip, dim=1)

            # Average TTA probabilities
            avg_probabilities = (probabilities + probabilities_hflip + probabilities_vflip) / 3.0
            predictions = avg_probabilities.argmax(dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(avg_probabilities.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    labels_np = np.array(all_labels)
    predictions_np = np.array(all_predictions)
    probabilities_np = np.array(all_probabilities)

    accuracy = accuracy_score(labels_np, predictions_np)
    balanced_accuracy = balanced_accuracy_score(labels_np, predictions_np)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_np, predictions_np, average="macro", zero_division=0
    )

    try:
        auc = roc_auc_score(labels_np, probabilities_np, multi_class="ovr", average="macro")
    except ValueError:
        auc = np.nan

    return {
        "loss": epoch_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "labels": labels_np,
        "predictions": predictions_np,
        "probabilities": probabilities_np
    }

### Cell 12 — Train local EfficientNet-B4

In [ ]:
# Cell 12: Local/Centralized training

history = {
    "train_loss": [], "train_accuracy": [], "train_bin_accuracy": [],
    "val_loss": [], "val_accuracy": [], "val_f1": [], "val_auc": [],
    "val_bin_accuracy": [], "val_bin_f1": [], "val_bin_auc": []
}

best_f1 = -1.0
best_epoch = -1
best_model_state = None
patience_counter = 0
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    train_loss, train_acc, train_bin_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_metrics = evaluate_model(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["train_bin_accuracy"].append(train_bin_acc)
    
    history["val_loss"].append(val_metrics["loss"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    history["val_f1"].append(val_metrics["f1"])
    history["val_auc"].append(val_metrics["auc"])
    
    history["val_bin_accuracy"].append(val_metrics["bin_accuracy"])
    history["val_bin_f1"].append(val_metrics["bin_f1"])
    history["val_bin_auc"].append(val_metrics["bin_auc"])

    current_lr = optimizer.param_groups[0]["lr"]

    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] "
          f"| Loss: {train_loss:.3f} "
          f"| 5-cls Acc: {train_acc:.3f} / {val_metrics['accuracy']:.3f} "
          f"| Bin Acc: {train_bin_acc:.3f} / {val_metrics['bin_accuracy']:.3f} "
          f"| 5-cls F1: {val_metrics['f1']:.3f} "
          f"| Bin F1: {val_metrics['bin_f1']:.3f} "
          f"| Time: {time.time()-epoch_start:.1f}s")

    # Select best model using Binary F1 for higher task priority (or Macro F1)
    # We will track best_model by 5-class Macro-F1, but report Binary.
    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, MODEL_DIR / "local_efficientnet_b4_best.pth")
        patience_counter = 0
        print(f"   --> New best model (5-cls F1={best_f1:.4f} | Bin F1={val_metrics['bin_f1']:.4f})")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

training_time = time.time() - start_time
print("\n" + "=" * 70)
print("LOCAL TRAINING COMPLETE")
print("=" * 70)
print("Best epoch:", best_epoch)
print("Best validation 5-cls Macro-F1:", best_f1)
print("Training time:", round(training_time / 60, 2), "minutes")

### Cell 13 — Load best model

In [ ]:
# Cell 13: Load best checkpoint

best_model_state = torch.load(
    MODEL_DIR / "local_efficientnet_b4_best.pth",
    map_location=DEVICE
)

model.load_state_dict(
    best_model_state
)

model = model.to(DEVICE)

print("Best local model loaded")

### Cell 14 — Plot training history

In [ ]:
# Cell 14: Plot training history

epochs_completed = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(epochs_completed, history["train_loss"], label="Train")
axes[0, 0].plot(epochs_completed, history["val_loss"], label="Validation")
axes[0, 0].set_title("Loss")
axes[0, 0].legend()

# 5-Class Accuracy & F1
axes[0, 1].plot(epochs_completed, history["train_accuracy"], label="Train Acc")
axes[0, 1].plot(epochs_completed, history["val_accuracy"], label="Val Acc")
axes[0, 1].plot(epochs_completed, history["val_f1"], label="Val F1", linestyle='--')
axes[0, 1].set_title("5-Class Accuracy & Macro-F1")
axes[0, 1].legend()

# Binary Accuracy
axes[1, 0].plot(epochs_completed, history["train_bin_accuracy"], label="Train Bin Acc")
axes[1, 0].plot(epochs_completed, history["val_bin_accuracy"], label="Val Bin Acc")
axes[1, 0].set_title("Binary Accuracy (No DR vs DR)")
axes[1, 0].legend()

# Binary AUC & F1
axes[1, 1].plot(epochs_completed, history["val_bin_f1"], label="Val Bin F1")
axes[1, 1].plot(epochs_completed, history["val_bin_auc"], label="Val Bin AUC")
axes[1, 1].set_title("Binary F1 and AUC")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(RESULT_DIR / "local_training_history.png", dpi=300, bbox_inches="tight")
plt.show()

### Cell 15 — Validation results

In [ ]:
# Cell 15: Final validation evaluation

val_metrics = evaluate_model(model, val_loader, criterion, DEVICE)

print("=" * 70)
print(f"{'LOCAL VALIDATION RESULTS':^70}")
print("=" * 70)
print(f"{'Metric':<25} | {'5-Class (0,1,2,3,4)':<20} | {'Binary (No-DR vs DR)':<20}")
print("-" * 70)
print(f"{'Accuracy':<25} | {val_metrics['accuracy']:.4f}{'':<14} | {val_metrics['bin_accuracy']:.4f}")
print(f"{'Balanced Accuracy':<25} | {val_metrics['balanced_accuracy']:.4f}{'':<14} | {val_metrics['bin_balanced_accuracy']:.4f}")
print(f"{'Macro Precision':<25} | {val_metrics['precision']:.4f}{'':<14} | {val_metrics['bin_precision']:.4f}")
print(f"{'Macro Recall':<25} | {val_metrics['recall']:.4f}{'':<14} | {val_metrics['bin_recall']:.4f}")
print(f"{'Macro F1':<25} | {val_metrics['f1']:.4f}{'':<14} | {val_metrics['bin_f1']:.4f}")
print(f"{'Macro AUC':<25} | {val_metrics['auc']:.4f}{'':<14} | {val_metrics['bin_auc']:.4f}")
print("=" * 70)

### Cell 16 — Validation classification report

In [ ]:
# Cell 16: Validation classification report

class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

print(
    classification_report(
        val_metrics["labels"],
        val_metrics["predictions"],
        labels=[0, 1, 2, 3, 4],
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

### Cell 17 — Validation confusion matrix

In [ ]:
# Cell 17: Validation confusion matrix

cm = confusion_matrix(
    val_metrics["labels"],
    val_metrics["predictions"],
    labels=[0, 1, 2, 3, 4]
)

plt.figure(figsize=(8, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Local Model — Validation Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()

plt.savefig(
    RESULT_DIR / "local_validation_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Cell 18 — Final official test evaluation

**This is the first time we evaluate the official 103-image test set.**

In [ ]:
# Cell 18: Official test evaluation

test_metrics = evaluate_model(model, test_loader, criterion, DEVICE)

print("=" * 70)
print(f"{'LOCAL MODEL - OFFICIAL TEST RESULTS':^70}")
print("=" * 70)
print(f"{'Metric':<25} | {'5-Class (0,1,2,3,4)':<20} | {'Binary (No-DR vs DR)':<20}")
print("-" * 70)
print(f"{'Accuracy':<25} | {test_metrics['accuracy']:.4f}{'':<14} | {test_metrics['bin_accuracy']:.4f}")
print(f"{'Balanced Accuracy':<25} | {test_metrics['balanced_accuracy']:.4f}{'':<14} | {test_metrics['bin_balanced_accuracy']:.4f}")
print(f"{'Macro Precision':<25} | {test_metrics['precision']:.4f}{'':<14} | {test_metrics['bin_precision']:.4f}")
print(f"{'Macro Recall':<25} | {test_metrics['recall']:.4f}{'':<14} | {test_metrics['bin_recall']:.4f}")
print(f"{'Macro F1':<25} | {test_metrics['f1']:.4f}{'':<14} | {test_metrics['bin_f1']:.4f}")
print(f"{'Macro AUC':<25} | {test_metrics['auc']:.4f}{'':<14} | {test_metrics['bin_auc']:.4f}")
print("=" * 70)

### Cell 19 — Official test classification report

In [ ]:
# Cell 19: Test classification report

print(
    classification_report(
        test_metrics["labels"],
        test_metrics["predictions"],
        labels=[0, 1, 2, 3, 4],
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

### Cell 20 — Official test confusion matrix

In [ ]:
# Cell 20: Test confusion matrix

cm_test = confusion_matrix(
    test_metrics["labels"],
    test_metrics["predictions"],
    labels=[0, 1, 2, 3, 4]
)

plt.figure(figsize=(8, 7))

sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Local EfficientNet-B4 — Official Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()

plt.savefig(
    RESULT_DIR / "local_test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Cell 21 — Save local results

In [ ]:
# Cell 21: Save local results

local_results = {
    "model": "EfficientNet-B4",
    "dataset": "IDRiD Disease Grading",
    "task": "5-class & Binary classification",
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_configured": NUM_EPOCHS,
    "best_epoch": best_epoch,
    "best_val_5cls_macro_f1": float(best_f1),
    "validation": {
        "5_class": {k: float(v) for k, v in val_metrics.items() if not k.startswith("bin_") and isinstance(v, (float, np.float32, np.float64)) and k not in ["loss", "labels", "predictions", "probabilities"]},
        "binary": {k: float(v) for k, v in val_metrics.items() if k.startswith("bin_") and isinstance(v, (float, np.float32, np.float64))}
    },
    "official_test": {
        "5_class": {k: float(v) for k, v in test_metrics.items() if not k.startswith("bin_") and isinstance(v, (float, np.float32, np.float64)) and k not in ["loss", "labels", "predictions", "probabilities"]},
        "binary": {k: float(v) for k, v in test_metrics.items() if k.startswith("bin_") and isinstance(v, (float, np.float32, np.float64))}
    }
}

with open(RESULT_DIR / "local_efficientnet_b4_results.json", "w") as f:
    json.dump(local_results, f, indent=4)

print("Local results saved")

### Cell 22 — Check whether 224 is good enough

Before moving to FedAvg, **do not automatically assume 224×224 is your final preprocessing**.

At this point, look at:

```text
Validation Accuracy
Validation Macro-F1
Validation AUC
Test Accuracy
Test Macro-F1
Test AUC
```

For your research objective, the next experiment should be **512×512 local EfficientNet-B4**, using the same train/validation split, before deciding which resolution to carry into FedAvg.

The decision rule should be based primarily on **validation Macro-F1/AUC**, not by repeatedly checking the official test set.

Once you have the output of the local training cells—especially **validation/test accuracy, Macro-F1, AUC, and the confusion matrix**—we can set the best local configuration and move to **Stage 3: FedAvg**.